# WignerFlow training on Colab
Runtime > Change runtime type > GPU (A100/L40S if available, otherwise T4).

Needs once on your Google Drive: `MyDrive/WignerFlow/data/argon/npy/` (the 5.8 GB `npy` folder from Hoffman2 or the local `data/argon/npy`).
Outputs and checkpoints go to `MyDrive/WignerFlow/results/...`. Training resumes from `ckpt.pt`, so a disconnected session can simply be re-run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BRANCH = 'v1.0-dev'
!git clone -b $BRANCH https://github.com/yaniguan/WignerFlow.git /content/WignerFlow 2>/dev/null || git -C /content/WignerFlow pull
%cd /content/WignerFlow
!pip install -q e3nn h5py pyyaml scipy matplotlib tensorboard
!pip install -q -e . --no-deps
# data and results live on Drive (symlinks keep the config paths unchanged)
!mkdir -p /content/drive/MyDrive/WignerFlow/data /content/drive/MyDrive/WignerFlow/results
!rm -rf data results && ln -s /content/drive/MyDrive/WignerFlow/data data && ln -s /content/drive/MyDrive/WignerFlow/results results
!ls data/argon/npy | head -3 && nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Jobs to run, in order. Each is one config; name = file in configs/argon/ (make_configs.py output is in git).
JOBS = ['B_S3a_seed6']          # edit, e.g. ['B_plain_seed2', 'B_S2_seed1', ...]
import subprocess, os
for j in JOBS:
    cfg = f'configs/argon/{j}.yaml'
    print('==', j)
    r = subprocess.run(['python', '-m', 'equitraj.train', '--config', cfg])
    print('exit', r.returncode)

In [ ]:
# After training: pick the checkpoint and run the 10^4-step test evaluation (same commands as on the cluster).
RUN = 'results/argon/B_S3a_seed6'
!python scripts/select_checkpoint.py --run $RUN --steps 4000 --starts 4
!python scripts/evaluate.py --run $RUN --ckpt selected.pt --steps 10000 --seeds 5 --starts 3